In [0]:
# 🎉 RESUMEN DEL PROYECTO

print("="*80)
print("🏆 PROYECTO COMPLETADO: CREDIT RISK SCORING")
print("="*80)
# Hice muchos cambios
print("\n🎯 ARQUITECTURA MEDALLION IMPLEMENTADA:")
print("   🥉 Bronze: workspace.bronze.credit_risk_raw")
print("   🥈 Silver: workspace.silver.credit_risk_cleaned")
print("   🥇 Gold:   workspace.gold.credit_risk_train/val/test")
print("            workspace.gold.credit_risk_scoring")

print("\n🤖 MODELOS ENTRENADOS Y COMPARADOS:")
print("   1. Logistic Regression")
print("   2. Random Forest")
print("   3. XGBoost")
print("   4. LightGBM")
print("   5. CatBoost")

print(f"\n🏆 MEJOR MODELO: {best_run['model']}")
print(f"   AUC en Test: {best_run['test_auc']:.4f}")

print("\n📊 RESULTADOS FINALES:")
print(f"   Total de clientes con scoring: {len(scoring_df):,}")
print("   Categorías de riesgo:")
for cat, count in risk_counts.items():
    print(f"      {cat}: {count:,} clientes ({count/len(scoring_df)*100:.1f}%)")

print("\n📦 TABLAS CREADAS:")
print("   - workspace.bronze.credit_risk_raw (datos originales)")
print("   - workspace.silver.credit_risk_cleaned (datos limpios)")
print("   - workspace.gold.credit_risk_train (datos entrenamiento)")
print("   - workspace.gold.credit_risk_val (datos validación)")
print("   - workspace.gold.credit_risk_test (datos test)")
print("   - workspace.gold.credit_risk_scoring (scoring final por cliente)")

print("\n🔑 EXPERIMENTO MLFLOW:")
print("   /Users/yltabord@gmail.com/credit_risk_scoring")
print(f"   Mejor Run ID: {best_run_id}")

print("\n✅ PROYECTO COMPLETADO EXITOSAMENTE")
print("="*80)

In [0]:
# COMPARACIÓN DE TODOS LOS MODELOS

import mlflow

from mlflow.tracking import MlflowClient
import pandas as pd
#new chance

print("="*80)
print("🏆 COMPARACIÓN DE MODELOS - RANKING POR AUC")
print("="*80)

# Buscar todos los runs del experimento
client = MlflowClient()
experiment = mlflow.get_experiment_by_name("/Users/yltabord@gmail.com/credit_risk_scoring")
runs = client.search_runs(experiment.experiment_id)

# Extraer métricas de cada run
results = []
for run in runs:
    run_data = {
        'run_id': run.info.run_id,
        'model': run.data.params.get('model_type', 'Unknown'),
        'test_auc': run.data.metrics.get('test_auc', 0),
        'test_accuracy': run.data.metrics.get('test_accuracy', 0),
        'test_precision': run.data.metrics.get('test_precision', 0),
        'test_recall': run.data.metrics.get('test_recall', 0),
        'test_f1': run.data.metrics.get('test_f1', 0),
        'training_time': run.data.metrics.get('training_time_seconds', 0)
    }
    results.append(run_data)

# Crear DataFrame y ordenar por AUC
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('test_auc', ascending=False).reset_index(drop=True)

# Formatear para visualización
comparison_df['rank'] = range(1, len(comparison_df) + 1)
comparison_df = comparison_df[['rank', 'model', 'test_auc', 'test_accuracy', 
                                'test_precision', 'test_recall', 'test_f1', 'training_time']]

print("\n📊 Resultados de todos los modelos (ordenados por Test AUC):\n")
display(comparison_df.style.background_gradient(subset=['test_auc'], cmap='RdYlGn'))

# Identificar el mejor modelo
best_run = comparison_df.iloc[0]
print("\n" + "="*80)
print("🎖️  MEJOR MODELO SELECCIONADO")
print("="*80)
print(f"\n🥇 Modelo: {best_run['model']}")
print(f"\n📊 Métricas en Test Set:")
print(f"   AUC:       {best_run['test_auc']:.4f}")
print(f"   Accuracy:  {best_run['test_accuracy']:.4f}")
print(f"   Precision: {best_run['test_precision']:.4f}")
print(f"   Recall:    {best_run['test_recall']:.4f}")
print(f"   F1 Score:  {best_run['test_f1']:.4f}")
print(f"\n⏱️  Tiempo de entrenamiento: {best_run['training_time']:.2f} segundos")

# Guardar el mejor run_id para uso posterior
best_run_id = best_run['run_id']
print(f"\n🔑 Run ID: {best_run_id}")

In [0]:
# Visualización de comparación de modelos

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📊 Comparación de Modelos de Clasificación', fontsize=18, fontweight='bold')

models = comparison_df['model'].tolist()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

# 1. AUC Comparison
axes[0, 0].barh(models, comparison_df['test_auc'], color=colors, alpha=0.8)
axes[0, 0].set_xlabel('AUC Score', fontweight='bold')
axes[0, 0].set_title('AUC (Area Under ROC Curve)', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim(0, 1)
axes[0, 0].grid(axis='x', alpha=0.3)
for i, v in enumerate(comparison_df['test_auc']):
    axes[0, 0].text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')

# 2. Accuracy Comparison
axes[0, 1].barh(models, comparison_df['test_accuracy'], color=colors, alpha=0.8)
axes[0, 1].set_xlabel('Accuracy', fontweight='bold')
axes[0, 1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlim(0, 1)
axes[0, 1].grid(axis='x', alpha=0.3)
for i, v in enumerate(comparison_df['test_accuracy']):
    axes[0, 1].text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')

# 3. F1 Score Comparison
axes[1, 0].barh(models, comparison_df['test_f1'], color=colors, alpha=0.8)
axes[1, 0].set_xlabel('F1 Score', fontweight='bold')
axes[1, 0].set_title('F1 Score', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim(0, 1)
axes[1, 0].grid(axis='x', alpha=0.3)
for i, v in enumerate(comparison_df['test_f1']):
    axes[1, 0].text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')

# 4. Training Time Comparison
axes[1, 1].barh(models, comparison_df['training_time'], color=colors, alpha=0.8)
axes[1, 1].set_xlabel('Tiempo (segundos)', fontweight='bold')
axes[1, 1].set_title('Tiempo de Entrenamiento', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)
for i, v in enumerate(comparison_df['training_time']):
    axes[1, 1].text(v + 0.5, i, f'{v:.1f}s', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [0]:
# Cargar el mejor modelo y generar matriz de confusión + ROC curve

from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el mejor modelo
best_model_uri = f"runs:/{best_run_id}/model"
best_model = mlflow.sklearn.load_model(best_model_uri)

print(f"✅ Mejor modelo cargado: {best_run['model']}")

# Predicciones en test set
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

# Crear visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'🏆 Evaluación del Mejor Modelo: {best_run["model"]}', fontsize=16, fontweight='bold')

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
axes[0].set_title('Matriz de Confusión', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor Real', fontweight='bold')
axes[0].set_xlabel('Predicción', fontweight='bold')

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title('Curva ROC', fontsize=14, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Visualizaciones generadas")

In [0]:
# SCORING DE RIESGO PARA TODOS LOS CLIENTES

print("="*80)
print("🎯 GENERANDO SCORING DE RIESGO DE CRÉDITO")
print("="*80)

# Cargar todos los datos (completos) de Gold
all_data = spark.table("workspace.silver.credit_risk_cleaned").toPandas()

print(f"\n📊 Total de clientes: {len(all_data):,}")

# Preparar datos (aplicar las mismas transformaciones)
X_all = all_data.drop('loan_status', axis=1)
y_all = all_data['loan_status']

# Codificar y escalar
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler

categorical_cols = X_all.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()

# Aplicar codificación
encoder = TargetEncoder(cols=categorical_cols)
X_all_encoded = encoder.fit_transform(X_all, y_all)

# Normalizar
scaler = StandardScaler()
X_all_scaled = X_all_encoded.copy()
X_all_scaled[numeric_cols] = scaler.fit_transform(X_all_encoded[numeric_cols])

print("✓ Datos preparados")

# Generar predicciones
print("\n🔮 Generando scores de riesgo...")
risk_probabilities = best_model.predict_proba(X_all_scaled)[:, 1]  # Probabilidad de default
risk_predictions = best_model.predict(X_all_scaled)

# Crear categorías de riesgo basadas en probabilidad
def categorize_risk(prob):
    if prob < 0.3:
        return 'Bajo'
    elif prob < 0.6:
        return 'Medio'
    else:
        return 'Alto'

risk_categories = [categorize_risk(p) for p in risk_probabilities]

# Crear DataFrame de resultados
scoring_df = pd.DataFrame({
    'customer_id': range(1, len(all_data) + 1),
    'risk_score': risk_probabilities,
    'risk_category': risk_categories,
    'default_prediction': risk_predictions,
    'actual_status': y_all
})

# Redondear score
scoring_df['risk_score'] = scoring_df['risk_score'].round(4)

print("✓ Scoring completado")

# Mostrar estadísticas
print("\n" + "="*80)
print("📊 DISTRIBUCIÓN DE CATEGORÍAS DE RIESGO")
print("="*80)
risk_dist = scoring_df['risk_category'].value_counts().sort_index()
print(risk_dist)
print(f"\nTotal: {len(scoring_df):,} clientes")

# Guardar resultados
print("\n💾 Guardando resultados...")

# Convertir a Spark DataFrame y guardar
scoring_spark_df = spark.createDataFrame(scoring_df)
scoring_spark_df.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_scoring")

print("✅ Tabla de scoring creada: workspace.gold.credit_risk_scoring")

# Mostrar muestra
print("\n" + "="*80)
print("👁️  MUESTRA DE RESULTADOS (primeros 20 clientes)")
print("="*80)
display(scoring_df.head(20))

In [0]:
# Visualización final del scoring

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🎯 Análisis de Scoring de Riesgo de Crédito', fontsize=18, fontweight='bold')

# 1. Distribución de Risk Scores
axes[0, 0].hist(scoring_df['risk_score'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Risk Score (Probabilidad de Default)', fontweight='bold')
axes[0, 0].set_ylabel('Frecuencia', fontweight='bold')
axes[0, 0].set_title('Distribución de Scores de Riesgo', fontsize=14, fontweight='bold')
axes[0, 0].axvline(x=0.3, color='green', linestyle='--', label='Umbral Bajo-Medio', linewidth=2)
axes[0, 0].axvline(x=0.6, color='red', linestyle='--', label='Umbral Medio-Alto', linewidth=2)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Distribución de Categorías de Riesgo
risk_counts = scoring_df['risk_category'].value_counts().sort_index()
colors_risk = ['green', 'orange', 'red']
axes[0, 1].bar(risk_counts.index, risk_counts.values, color=colors_risk, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Categoría de Riesgo', fontweight='bold')
axes[0, 1].set_ylabel('Número de Clientes', fontweight='bold')
axes[0, 1].set_title('Distribución por Categoría de Riesgo', fontsize=14, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(risk_counts.values):
    axes[0, 1].text(i, v + 100, str(v), ha='center', fontweight='bold')

# 3. Pie Chart de Categorías
axes[1, 0].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
               colors=colors_risk, startangle=90, textprops={'fontweight': 'bold'})
axes[1, 0].set_title('Proporción de Categorías de Riesgo', fontsize=14, fontweight='bold')

# 4. Comparación: Predicciones vs Realidad
actual_default = scoring_df[scoring_df['actual_status'] == 1]['risk_category'].value_counts().sort_index()
actual_no_default = scoring_df[scoring_df['actual_status'] == 0]['risk_category'].value_counts().sort_index()

x = np.arange(len(risk_counts.index))
width = 0.35

axes[1, 1].bar(x - width/2, actual_no_default.values, width, label='No Default Real', 
               color='green', alpha=0.7, edgecolor='black')
axes[1, 1].bar(x + width/2, actual_default.values, width, label='Default Real', 
               color='red', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Categoría de Riesgo Predicha', fontweight='bold')
axes[1, 1].set_ylabel('Número de Clientes', fontweight='bold')
axes[1, 1].set_title('Validación: Categorías vs Estado Real', fontsize=14, fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(risk_counts.index)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualizaciones finales generadas")

In [0]:
# Cargar datos procesados para entrenamiento

train_df = spark.table("workspace.gold.credit_risk_train").toPandas()
val_df = spark.table("workspace.gold.credit_risk_val").toPandas()
test_df = spark.table("workspace.gold.credit_risk_test").toPandas()

X_train = train_df.drop('loan_status', axis=1)
y_train = train_df['loan_status']

X_val = val_df.drop('loan_status', axis=1)
y_val = val_df['loan_status']

X_test = test_df.drop('loan_status', axis=1)
y_test = test_df['loan_status']

print("✅ Datos cargados para entrenamiento:")
print(f"   Train: {X_train.shape}")
print(f"   Val:   {X_val.shape}")
print(f"   Test:  {X_test.shape}")

In [0]:
# Configuración de MLflow
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# Configurar experimento
mlflow.set_experiment("/Users/yltabord@gmail.com/credit_risk_scoring")

print("✅ Experimento MLflow configurado: /Users/yltabord@gmail.com/credit_risk_scoring")

# Función para evaluar modelos
def evaluate_model(model, X_val, y_val, X_test, y_test):
    """Evaluar modelo en validation y test sets"""
    # Predicciones
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    
    y_test_pred = model.predict(X_test)
    y_test_proba = model.predict_proba(X_test)[:, 1]
    
    # Métricas
    metrics = {
        'val_auc': roc_auc_score(y_val, y_val_proba),
        'val_accuracy': accuracy_score(y_val, y_val_pred),
        'val_precision': precision_score(y_val, y_val_pred),
        'val_recall': recall_score(y_val, y_val_pred),
        'val_f1': f1_score(y_val, y_val_pred),
        'test_auc': roc_auc_score(y_test, y_test_proba),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'test_precision': precision_score(y_test, y_test_pred),
        'test_recall': recall_score(y_test, y_test_pred),
        'test_f1': f1_score(y_test, y_test_pred)
    }
    
    return metrics

print("✓ Funciones de evaluación configuradas")

In [0]:
# MODELO 1: Logistic Regression

from sklearn.linear_model import LogisticRegression
import time

print("="*70)
print("🔵 MODELO 1: LOGISTIC REGRESSION")
print("="*70)

with mlflow.start_run(run_name="Logistic_Regression"):
    start_time = time.time()
    
    # Entrenar modelo
    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',  # Manejar desbalance
        C=1.0
    )
    
    lr_model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Evaluar
    metrics = evaluate_model(lr_model, X_val, y_val, X_test, y_test)
    
    # Log en MLflow
    mlflow.log_params({
        'model_type': 'Logistic Regression',
        'max_iter': 1000,
        'C': 1.0,
        'class_weight': 'balanced'
    })
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric('training_time_seconds', training_time)
    
    # Log modelo
    mlflow.sklearn.log_model(lr_model, "model")
    
    print(f"\n✅ Entrenamiento completado en {training_time:.2f}s")
    print(f"\n🏆 Resultados:")
    print(f"   Validation AUC: {metrics['val_auc']:.4f}")
    print(f"   Test AUC:       {metrics['test_auc']:.4f}")
    print(f"   Test Accuracy:  {metrics['test_accuracy']:.4f}")
    print(f"   Test F1:        {metrics['test_f1']:.4f}")

In [0]:
# MODELO 2: Random Forest

from sklearn.ensemble import RandomForestClassifier
import time

print("="*70)
print("🌳 MODELO 2: RANDOM FOREST")
print("="*70)

with mlflow.start_run(run_name="Random_Forest"):
    start_time = time.time()
    
    # Entrenar modelo
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    
    rf_model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Evaluar
    metrics = evaluate_model(rf_model, X_val, y_val, X_test, y_test)
    
    # Log en MLflow
    mlflow.log_params({
        'model_type': 'Random Forest',
        'n_estimators': 200,
        'max_depth': 15,
        'min_samples_split': 10,
        'min_samples_leaf': 4,
        'class_weight': 'balanced'
    })
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric('training_time_seconds', training_time)
    
    # Log modelo
    mlflow.sklearn.log_model(rf_model, "model")
    
    print(f"\n✅ Entrenamiento completado en {training_time:.2f}s")
    print(f"\n🏆 Resultados:")
    print(f"   Validation AUC: {metrics['val_auc']:.4f}")
    print(f"   Test AUC:       {metrics['test_auc']:.4f}")
    print(f"   Test Accuracy:  {metrics['test_accuracy']:.4f}")
    print(f"   Test F1:        {metrics['test_f1']:.4f}")

In [0]:
# MODELO 3: XGBoost

import xgboost as xgb
import time

print("="*70)
print("🚀 MODELO 3: XGBOOST")
print("="*70)

with mlflow.start_run(run_name="XGBoost"):
    start_time = time.time()
    
    # Calcular scale_pos_weight para manejar desbalance
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    # Entrenar modelo
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc',
        use_label_encoder=False
    )
    
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )
    training_time = time.time() - start_time
    
    # Evaluar
    metrics = evaluate_model(xgb_model, X_val, y_val, X_test, y_test)
    
    # Log en MLflow
    mlflow.log_params({
        'model_type': 'XGBoost',
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': scale_pos_weight
    })
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric('training_time_seconds', training_time)
    
    # Log modelo
    mlflow.xgboost.log_model(xgb_model, "model")
    
    print(f"\n✅ Entrenamiento completado en {training_time:.2f}s")
    print(f"\n🏆 Resultados:")
    print(f"   Validation AUC: {metrics['val_auc']:.4f}")
    print(f"   Test AUC:       {metrics['test_auc']:.4f}")
    print(f"   Test Accuracy:  {metrics['test_accuracy']:.4f}")
    print(f"   Test F1:        {metrics['test_f1']:.4f}")

In [0]:
# MODELO 4: LightGBM

import lightgbm as lgb
import time

print("="*70)
print("⚡ MODELO 4: LIGHTGBM")
print("="*70)

with mlflow.start_run(run_name="LightGBM"):
    start_time = time.time()
    
    # Calcular scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    # Entrenar modelo
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        verbose=-1
    )
    
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    training_time = time.time() - start_time
    
    # Evaluar
    metrics = evaluate_model(lgb_model, X_val, y_val, X_test, y_test)
    
    # Log en MLflow
    mlflow.log_params({
        'model_type': 'LightGBM',
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': scale_pos_weight
    })
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric('training_time_seconds', training_time)
    
    # Log modelo
    mlflow.lightgbm.log_model(lgb_model, "model")
    
    print(f"\n✅ Entrenamiento completado en {training_time:.2f}s")
    print(f"\n🏆 Resultados:")
    print(f"   Validation AUC: {metrics['val_auc']:.4f}")
    print(f"   Test AUC:       {metrics['test_auc']:.4f}")
    print(f"   Test Accuracy:  {metrics['test_accuracy']:.4f}")
    print(f"   Test F1:        {metrics['test_f1']:.4f}")

In [0]:
# MODELO 5: CatBoost

from catboost import CatBoostClassifier
import time

print("="*70)
print("🐈 MODELO 5: CATBOOST")
print("="*70)

with mlflow.start_run(run_name="CatBoost"):
    start_time = time.time()
    
    # Calcular scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    # Entrenar modelo
    cat_model = CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        verbose=False,
        early_stopping_rounds=50
    )
    
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    training_time = time.time() - start_time
    
    # Evaluar
    metrics = evaluate_model(cat_model, X_val, y_val, X_test, y_test)
    
    # Log en MLflow
    mlflow.log_params({
        'model_type': 'CatBoost',
        'iterations': 300,
        'depth': 6,
        'learning_rate': 0.05,
        'scale_pos_weight': scale_pos_weight
    })
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric('training_time_seconds', training_time)
    
    # Log modelo
    mlflow.catboost.log_model(cat_model, "model")
    
    print(f"\n✅ Entrenamiento completado en {training_time:.2f}s")
    print(f"\n🏆 Resultados:")
    print(f"   Validation AUC: {metrics['val_auc']:.4f}")
    print(f"   Test AUC:       {metrics['test_auc']:.4f}")
    print(f"   Test Accuracy:  {metrics['test_accuracy']:.4f}")
    print(f"   Test F1:        {metrics['test_f1']:.4f}")

In [0]:
# CAPA SILVER: Limpieza y Transformación de Datos

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, mean, stddev, percentile_approx

# Leer datos de Bronze
df_bronze = spark.table("workspace.bronze.credit_risk_raw")

print("✅ Iniciando transformación Silver...")
print(f"Filas iniciales: {df_bronze.count():,}")

# 1. Manejo de valores faltantes
print("\n1️⃣ Imputando valores faltantes...")

# Calcular la media de loan_int_rate para imputar
avg_int_rate = df_bronze.select(mean("loan_int_rate")).collect()[0][0]
avg_emp_length = df_bronze.select(mean("person_emp_length")).collect()[0][0]

df_silver = df_bronze \
    .withColumn("loan_int_rate", 
                when(col("loan_int_rate").isNull(), avg_int_rate)
                .otherwise(col("loan_int_rate"))) \
    .withColumn("person_emp_length", 
                when(col("person_emp_length").isNull(), avg_emp_length)
                .otherwise(col("person_emp_length")))

print(f"   ✓ Tasa de interés: imputada con media = {avg_int_rate:.2f}%")
print(f"   ✓ Años de empleo: imputada con media = {avg_emp_length:.2f} años")

# 2. Remover outliers extremos
print("\n2️⃣ Removiendo outliers...")

# Filtrar edades razonables (18-100) y remover valores extremos
df_silver = df_silver \
    .filter((col("person_age") >= 18) & (col("person_age") <= 100)) \
    .filter(col("person_income") > 0) \
    .filter(col("person_emp_length") >= 0)

print(f"   ✓ Outliers removidos")
print(f"   Filas después de limpieza: {df_silver.count():,}")

# 3. Feature Engineering básico
print("\n3️⃣ Creando features adicionales...")

df_silver = df_silver \
    .withColumn("debt_to_income_ratio", col("loan_amnt") / col("person_income")) \
    .withColumn("income_per_age", col("person_income") / col("person_age")) \
    .withColumn("loan_to_income_ratio", col("loan_percent_income"))

print("   ✓ Features creadas: debt_to_income_ratio, income_per_age")

# Guardar en Silver
df_silver.write.mode("overwrite").saveAsTable("workspace.silver.credit_risk_cleaned")
print("\n✅ Tabla Silver creada: workspace.silver.credit_risk_cleaned")

display(df_silver.limit(10))

In [0]:
# CAPA GOLD: Preparación para Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from category_encoders import TargetEncoder
import pandas as pd

# Leer datos de Silver
df_silver = spark.table("workspace.silver.credit_risk_cleaned")
df_pd = df_silver.toPandas()

print("✅ Preparando datos para ML...")
print(f"Tamaño del dataset: {df_pd.shape}")

# Separar features y target
X = df_pd.drop('loan_status', axis=1)
y = df_pd['loan_status']

print(f"\nFeatures: {X.shape[1]}")
print(f"Target: {y.name}")
print(f"Distribución: {y.value_counts().to_dict()}")

# Identificar columnas categóricas y numéricas
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"\nColumnas categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"Columnas numéricas ({len(numeric_cols)}): {numeric_cols}")

# Codificar variables categóricas usando Target Encoding
print("\n🔄 Codificando variables categóricas con Target Encoding...")

encoder = TargetEncoder(cols=categorical_cols)
X_encoded = encoder.fit_transform(X, y)

print("✓ Codificación completada")

# Normalizar features numéricas
print("\n📊 Normalizando features numéricas...")

scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_encoded[numeric_cols])

print("✓ Normalización completada")

# Split: 70% train, 15% validation, 15% test
print("\n✂️ Dividiendo datos: 70% train / 15% val / 15% test...")

X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp  # 0.176 de 0.85 ≈ 15% del total
)

print(f"\n✓ Split completado:")
print(f"   Train: {X_train.shape[0]:,} muestras ({X_train.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"   Val:   {X_val.shape[0]:,} muestras ({X_val.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"   Test:  {X_test.shape[0]:,} muestras ({X_test.shape[0]/len(X_scaled)*100:.1f}%)")

# Guardar datos procesados
print("\n💾 Guardando datasets procesados...")

# Crear DataFrames de Spark y guardar
train_df = spark.createDataFrame(pd.concat([X_train, y_train], axis=1))
val_df = spark.createDataFrame(pd.concat([X_val, y_val], axis=1))
test_df = spark.createDataFrame(pd.concat([X_test, y_test], axis=1))

train_df.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_train")
val_df.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_val")
test_df.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_test")

print("✅ Tablas Gold creadas:")
print("   - workspace.gold.credit_risk_train")
print("   - workspace.gold.credit_risk_val")
print("   - workspace.gold.credit_risk_test")

In [0]:
# EXPLORATORY DATA ANALYSIS (EDA)
# Cargar datos de Bronze para análisis

df_bronze = spark.table("workspace.bronze.credit_risk_raw")
df_pd = df_bronze.toPandas()

print("="*60)
print("INFORMACIÓN DEL DATASET")
print("="*60)
print(f"\nForma del dataset: {df_pd.shape}")
print(f"Filas: {df_pd.shape[0]:,} | Columnas: {df_pd.shape[1]}")

print("\n" + "="*60)
print("TIPOS DE DATOS")
print("="*60)
print(df_pd.dtypes)

print("\n" + "="*60)
print("VALORES FALTANTES")
print("="*60)
missing = df_pd.isnull().sum()
missing_pct = (missing / len(df_pd)) * 100
missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores Faltantes': missing.values,
    'Porcentaje': missing_pct.values
})
print(missing_df[missing_df['Valores Faltantes'] > 0])

print("\n" + "="*60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
display(df_pd.describe())

In [0]:
# EDA - Distribución de la variable objetivo (loan_status)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo de clases
target_counts = df_pd['loan_status'].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], target_counts.values, color=['green', 'red'], alpha=0.7)
axes[0].set_title('Distribución de la Variable Objetivo', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontweight='bold')

# Porcentaje
axes[1].pie(target_counts.values, labels=['No Default', 'Default'], 
            autopct='%1.1f%%', colors=['green', 'red'], startangle=90)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n⚠️  Desbalance de clases detectado:")
print(f"   No Default: {target_counts[0]:,} ({target_counts[0]/len(df_pd)*100:.1f}%)")
print(f"   Default: {target_counts[1]:,} ({target_counts[1]/len(df_pd)*100:.1f}%)")
print(f"   Ratio: {target_counts[0]/target_counts[1]:.2f}:1")

In [0]:
# EDA - Correlación entre features numéricas

# Seleccionar solo columnas numéricas
numeric_cols = df_pd.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_pd[numeric_cols].corr()

# Correlogram
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlograma - Features Numéricas', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Correlación con el target
print("\n" + "="*60)
print("CORRELACIÓN CON LA VARIABLE OBJETIVO (loan_status)")
print("="*60)
target_corr = corr_matrix['loan_status'].sort_values(ascending=False)
print(target_corr)

In [0]:
# EDA - Visualizaciones de features vs target

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribución de Features por Estado de Default', fontsize=16, fontweight='bold')

# Edad
axes[0, 0].hist([df_pd[df_pd['loan_status']==0]['person_age'], 
                 df_pd[df_pd['loan_status']==1]['person_age']], 
                label=['No Default', 'Default'], bins=30, alpha=0.7, color=['green', 'red'])
axes[0, 0].set_xlabel('Edad')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Ingreso
axes[0, 1].hist([df_pd[df_pd['loan_status']==0]['person_income'], 
                 df_pd[df_pd['loan_status']==1]['person_income']], 
                label=['No Default', 'Default'], bins=30, alpha=0.7, color=['green', 'red'])
axes[0, 1].set_xlabel('Ingreso')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Monto del préstamo
axes[0, 2].hist([df_pd[df_pd['loan_status']==0]['loan_amnt'], 
                 df_pd[df_pd['loan_status']==1]['loan_amnt']], 
                label=['No Default', 'Default'], bins=30, alpha=0.7, color=['green', 'red'])
axes[0, 2].set_xlabel('Monto del Préstamo')
axes[0, 2].set_ylabel('Frecuencia')
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Tasa de interés
axes[1, 0].hist([df_pd[df_pd['loan_status']==0]['loan_int_rate'], 
                 df_pd[df_pd['loan_status']==1]['loan_int_rate']], 
                label=['No Default', 'Default'], bins=30, alpha=0.7, color=['green', 'red'])
axes[1, 0].set_xlabel('Tasa de Interés')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Porcentaje de ingreso en préstamo
axes[1, 1].hist([df_pd[df_pd['loan_status']==0]['loan_percent_income'], 
                 df_pd[df_pd['loan_status']==1]['loan_percent_income']], 
                label=['No Default', 'Default'], bins=30, alpha=0.7, color=['green', 'red'])
axes[1, 1].set_xlabel('% Ingreso en Préstamo')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

# Años de empleo
axes[1, 2].hist([df_pd[df_pd['loan_status']==0]['person_emp_length'], 
                 df_pd[df_pd['loan_status']==1]['person_emp_length']], 
                label=['No Default', 'Default'], bins=30, alpha=0.7, color=['green', 'red'])
axes[1, 2].set_xlabel('Años de Empleo')
axes[1, 2].set_ylabel('Frecuencia')
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# EDA - Tabla de tipos de features

feature_types = [
    ['person_age', 'Numérica (Entero)', 'Edad del solicitante'],
    ['person_income', 'Numérica (Entero)', 'Ingreso anual'],
    ['person_home_ownership', 'Categórica (Nominal)', 'Tipo de vivienda (RENT/OWN/MORTGAGE/OTHER)'],
    ['person_emp_length', 'Numérica (Float)', 'Años de empleo'],
    ['loan_intent', 'Categórica (Nominal)', 'Propósito del préstamo'],
    ['loan_grade', 'Categórica (Ordinal)', 'Grado del préstamo (A-G)'],
    ['loan_amnt', 'Numérica (Entero)', 'Monto del préstamo'],
    ['loan_int_rate', 'Numérica (Float)', 'Tasa de interés'],
    ['loan_status', 'Binaria (Target)', '0=No Default, 1=Default'],
    ['loan_percent_income', 'Numérica (Float)', '% del ingreso comprometido'],
    ['cb_person_default_on_file', 'Categórica (Binaria)', 'Historial de default (Y/N)'],
    ['cb_person_cred_hist_length', 'Numérica (Entero)', 'Longitud del historial crediticio']
]

feature_df = pd.DataFrame(feature_types, columns=['Feature', 'Tipo', 'Descripción'])
print("\n" + "="*80)
print("TABLA DE TIPOS DE FEATURES")
print("="*80)
display(feature_df)

In [0]:
%pip install xgboost lightgbm catboost optuna category_encoders imbalanced-learn

In [0]:
dbutils.library.restartPython()

In [0]:
# Imports necesarios
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.types import *
import mlflow
import mlflow.sklearn

print("✓ Librerías importadas correctamente")

## 📥 Descarga Automática del Dataset desde Kaggle

### Opción 1: Usar la API de Kaggle (Recomendado)

Para descargar automáticamente, necesitas:

1. **Crear una cuenta en Kaggle** (si no la tienes)
2. **Obtener tu API token**:
   - Ve a [kaggle.com/settings](https://www.kaggle.com/settings)
   - Scroll hasta "API" → Click "Create New Token"
   - Se descargará un archivo `kaggle.json`

3. **Subir las credenciales a Databricks**:
   ```python
   # Opción A: Usar Secrets (Producción)
   dbutils.secrets.put(scope="kaggle", key="username", string_value="tu_usuario")
   dbutils.secrets.put(scope="kaggle", key="key", string_value="tu_api_key")
   
   # Opción B: Variables de entorno (Desarrollo)
   import os
   os.environ['KAGGLE_USERNAME'] = 'tu_usuario'
   os.environ['KAGGLE_KEY'] = 'tu_api_key'
   ```

### Opción 2: Descarga Manual

Si prefieres descarga manual:
1. Ve a [kaggle.com/datasets/laotse/credit-risk-dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data)
2. Descarga `credit_risk_dataset.csv`
3. Súbelo a: `/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv`

---

**Ejecuta la celda siguiente para descargar automáticamente con la API de Kaggle** ⬇️

In [0]:
# Descarga automática del dataset desde Kaggle

import os
import subprocess

print("="*80)
print("📥 DESCARGA AUTOMÁTICA DEL DATASET DESDE KAGGLE")
print("="*80)

# Instalar Kaggle API
print("\n1️⃣ Instalando Kaggle API...")
%pip install -q kaggle

print("\n2️⃣ Configurando credenciales de Kaggle...")

# OPCIÓN 1: Usar credenciales desde variables de entorno (para desarrollo)
# Descomenta y reemplaza con tus credenciales:
# os.environ['KAGGLE_USERNAME'] = 'tu_usuario_kaggle'
# os.environ['KAGGLE_KEY'] = 'tu_api_key_kaggle'

# OPCIÓN 2: Usar Databricks Secrets (para producción)
# Descomenta si ya configuraste secrets:
# os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope="kaggle", key="username")
# os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope="kaggle", key="key")

# Verificar si las credenciales están configuradas
if 'KAGGLE_USERNAME' not in os.environ or 'KAGGLE_KEY' not in os.environ:
    print("\n⚠️ ATENCIÓN: Credenciales de Kaggle no configuradas")
    print("\nPor favor, configura tus credenciales de Kaggle en la celda anterior:")
    print("   1. Obtén tu API token desde: https://www.kaggle.com/settings")
    print("   2. Descomenta y completa las líneas de KAGGLE_USERNAME y KAGGLE_KEY")
    print("\nO descarga manualmente desde:")
    print("   https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data")
    raise Exception("Credenciales de Kaggle no configuradas")

print("✓ Credenciales configuradas")

# Crear directorio temporal para la descarga
temp_dir = "/tmp/kaggle_download"
os.makedirs(temp_dir, exist_ok=True)

print(f"\n3️⃣ Descargando dataset desde Kaggle...")
print("   Dataset: laotse/credit-risk-dataset")
print(f"   Destino temporal: {temp_dir}")

# Descargar dataset usando Kaggle API
try:
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "laotse/credit-risk-dataset", "-p", temp_dir, "--unzip"],
        capture_output=True,
        text=True,
        check=True
    )
    print("✓ Descarga completada")
except subprocess.CalledProcessError as e:
    print(f"❌ Error en la descarga: {e.stderr}")
    raise

# Crear el directorio del Volume si no existe
volume_path = "/Volumes/workspace/bronze/landing_zone"
print(f"\n4️⃣ Creando directorio en Volume: {volume_path}")
dbutils.fs.mkdirs(volume_path)

# Mover el archivo al Volume
source_file = f"{temp_dir}/credit_risk_dataset.csv"
dest_file = f"{volume_path}/credit_risk_dataset.csv"

print(f"\n5️⃣ Copiando archivo al Volume...")
print(f"   Desde: {source_file}")
print(f"   Hacia: {dest_file}")

# Copiar usando dbutils
dbutils.fs.cp(f"file://{source_file}", dest_file, recurse=False)

print("\n✅ Dataset descargado y guardado correctamente")
print(f"\n📍 Ubicación final: {dest_file}")
print("\n" + "="*80)

# Limpiar archivos temporales
import shutil
shutil.rmtree(temp_dir, ignore_errors=True)
print("✓ Archivos temporales eliminados")

In [0]:
# CAPA BRONZE: Carga de datos crudos desde Kaggle
# Primero, descarga el archivo CSV y súbelo a un Volume en Databricks
# Para este ejemplo, vamos a usar la URL directa del dataset

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Esquema del dataset de Kaggle
schema = StructType([
    StructField("person_age", IntegerType(), True),
    StructField("person_income", IntegerType(), True),
    StructField("person_home_ownership", StringType(), True),
    StructField("person_emp_length", DoubleType(), True),
    StructField("loan_intent", StringType(), True),
    StructField("loan_grade", StringType(), True),
    StructField("loan_amnt", IntegerType(), True),
    StructField("loan_int_rate", DoubleType(), True),
    StructField("loan_status", IntegerType(), True),  # Target: 0=no default, 1=default
    StructField("loan_percent_income", DoubleType(), True),
    StructField("cb_person_default_on_file", StringType(), True),
    StructField("cb_person_cred_hist_length", IntegerType(), True)
])

# IMPORTANTE: Primero debes subir el archivo CSV a un Volume en Databricks
# Por ejemplo: /Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv
# Ajusta esta ruta según tu configuración

csv_path = "/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv"

# Leer datos crudos
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(csv_path)

print(f"✓ Datos cargados: {df_bronze.count()} filas")

# Guardar en Bronze
df_bronze.write.mode("overwrite").saveAsTable("workspace.bronze.credit_risk_raw")
print("✓ Tabla Bronze creada: workspace.bronze.credit_risk_raw")

display(df_bronze.limit(10))